# Junção de tabelas:

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np


## 1. Presença por Deputado:

### **Métrica necessária**

- Número total de sessões deliberativas (por deputado)

- Número de presenças (por deputado)

- Taxa de presença (%)

| Dataset      | Coluna          | Para quê?                                                           |
| ------------ | --------------- | ------------------------------------------------------------------- |
| df_eventos   | `id`            | identificar cada evento (sessão)                                    |
| df_eventos   | `descricaoTipo` | filtrar somente “Sessão Deliberativa”                               |
| df_presencas | `id_evento`     | ligar presenças aos eventos                                         |
| df_presencas | `id_deputado`   | identificar o deputado                                              |
| df_presencas | `tipo_presenca` | classificar se foi **Presença**, **Ausência**, **Justificada** etc. |

### **Derivadas**

- total_sessoes

- total_presencas

- taxa_presenca = total_presencas / total_sessoes

### Criar colunas:

## 2. Sessões por Taxa de Votações Participadas:

### **Métricas necessárias**

- Sessões totais

- Votações possíveis

- Votações participadas

- Taxa de votação por deputado

| Dataset      | Coluna        | Para quê?                                         |
| ------------ | ------------- | ------------------------------------------------- |
| votacoes_all | `id`          | id da votação                                     |
| votacoes_all | `uriEvento`   | para saber qual evento originou a votação         |
| df_eventos   | `id`          | conectar votação ao evento                        |
| df_presencas | `id_deputado` | identificar deputado                              |
| df_presencas | `id_evento`   | saber em quais eventos o deputado esteve presente |

### **Derivadas**

- votacoes_possiveis = total de votações nos eventos que o deputado participou

- votacoes_participadas = quantidade de votações registradas como participação
(→ isso depende se você tem o dataset de votos por deputado. Se não tiver, você terá que assumir participação quando presença = presente.)

- taxa_votacao = votacoes_participadas / votacoes_possiveis


### Criar colunas:

## 3. Faltas Injustificadas vs Limite CLT:

### **Métrica necessária**

- Faltas contabilizadas como não justificadas

- Comparação com limite CLT (12 faltas por ano)

| Dataset      | Coluna          | Para quê?                             |
| ------------ | --------------- | ------------------------------------- |
| df_presencas | `tipo_presenca` | detectar “ausência sem justificativa” |
| df_presencas | `id_deputado`   | identificar deputado                  |

### **Derivadas**

- faltas_injustificadas = count(tipo_presenca == 'Ausência')

- (opcional) faltas_justificadas = count(tipo_presenca == 'Justificada')

### Criar colunas:

## 4. Ganho por Dia Trabalhado e por Votação:

### **Alinhar** 
id_deputado, salario_bruto


- ganho_por_dia_trabalhado = salario_bruto / total_presencas

- ganho_por_votacao_participada = salario_bruto / votacoes_participadas

### Criar colunas:

## 5. Desigualdade — Deputados VS Trabalhadores CLT:

- O dataset dos deputados (já acima)

- Um dataset CLT fictício ou real

- Derivada: ganho_por_dia_trabalhado (já calculada)

### Criar colunas:

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import plotly.express as px
import itertools

df_deputados = pd.read_json('../../data/processed/deputados.json') 
df_eventos = pd.read_csv('../../data/processed/eventos.csv')
df_presencas = pd.read_csv('../../data/processed/presencas.csv')
votacoes_all = pd.read_csv("../../data/processed/votacoes_2020-02_2025-10_todas.csv")

# Para df_eventos
print("Colunas de df_eventos:")
print(df_eventos.columns.tolist())

# Para df_presencas
print("\nColunas de df_presencas:")
print(df_presencas.columns.tolist())

# Para votacoes_all
print("\nColunas de votacoes_all:")
print(votacoes_all.columns.tolist())

# Para df_deputados
print("\nColunas de deputados:")
print(df_deputados.columns.tolist())

# A partir desses criei:
df_dep_presenca = pd.read_csv("../../data/graphs/df_dep_presenca.csv")
df_dep_votacao = pd.read_csv("../../data/graphs/df_dep_votacao.csv")
df_dep_injustificado = pd.read_csv("../../data/graphs/df_dep_injustificado.csv")

print("\nColunas de deputados:")
print(df_dep_presenca.columns.tolist())

print("\nColunas de deputados:")
print(df_dep_votacao.columns.tolist())

print("\nColunas de deputados:")
print(df_dep_injustificado.columns.tolist())


Colunas de df_eventos:
['id', 'uri', 'dataHoraInicio', 'dataHoraFim', 'situacao', 'descricaoTipo', 'descricao', 'localExterno', 'orgaos', 'localCamara', 'urlRegistro']

Colunas de df_presencas:
['id_evento', 'id_deputado', 'tipo_presenca', 'ano_origem']

Colunas de votacoes_all:
['id', 'uri', 'data', 'dataHoraRegistro', 'siglaOrgao', 'uriOrgao', 'uriEvento', 'proposicaoObjeto', 'uriProposicaoObjeto', 'descricao', 'aprovacao']

Colunas de deputados:
['id', 'uri', 'nome', 'siglaPartido', 'uriPartido', 'siglaUf', 'idLegislatura', 'urlFoto', 'email']

Colunas de deputados:
['id_deputado', 'qtd_presencas', 'total_sessoes', 'taxa_presenca', 'id', 'nome', 'siglaPartido', 'siglaUf', 'deputado']

Colunas de deputados:
['id_deputado', 'deputado', 'presencas', 'sessoes_total', 'votacoes_participadas', 'taxa_votacao']

Colunas de deputados:
['id_deputado', 'deputado', 'faltas_injustificadas']


In [2]:
import numpy as np

# base para ganhos: um deputado por linha, com presenças e votações
df_dep_ganhos = df_dep_votacao[[
    "id_deputado",
    "deputado",
    "presencas",
    "votacoes_participadas",
]].copy()

REMUNERACAO_BRUTA_REFERENCIA = 46000  # ajuste depois se quiser

# Ganho por Dia Trabalhado = remuneração / nº de presenças
df_dep_ganhos["ganho_por_dia_trabalhado"] = (
    REMUNERACAO_BRUTA_REFERENCIA /
    df_dep_ganhos["presencas"].replace(0, np.nan)
)

# Ganho por Votação Participada = remuneração / nº de votações participadas
df_dep_ganhos["ganho_por_votacao_participada"] = (
    REMUNERACAO_BRUTA_REFERENCIA /
    df_dep_ganhos["votacoes_participadas"].replace(0, np.nan)
)

# limpa infinitos (caso alguma coluna tenha ficado 0 e ainda gere inf)
df_dep_ganhos = df_dep_ganhos.replace([np.inf, -np.inf], np.nan)

# se quiser, pode remover linhas sem presenças/votações:
# df_dep_ganhos = df_dep_ganhos.dropna(
#     subset=["ganho_por_dia_trabalhado", "ganho_por_votacao_participada"]
# )

# salva pra usar no app
df_dep_ganhos.to_csv("../../data/graphs/df_dep_ganhos.csv", index=False)

df_dep_ganhos

,id_deputado,deputado,presencas,votacoes_participadas,ganho_por_dia_trabalhado,ganho_por_votacao_participada
0,62881,Danilo Forte,266,1827,172.932331,25.177887
1,66179,NaN,221,1248,208.144796,36.858974
2,66385,Julio Arcoverde,120,519,383.333333,88.631985
3,66828,Fausto Pinato,329,2028,139.817629,22.682446
4,67138,NaN,197,1093,233.502538,42.086002
...,...,...,...,...,...,...
890,230957,João Cury,38,149,1210.526316,308.724832
891,231911,Lenir de Assis,27,52,1703.703704,884.615385
892,233592,Paulo Lemos,8,14,5750.000000,3285.714286
893,233594,Rafael Fera,8,7,5750.000000,6571.428571


In [3]:
df_dep_votacao.head()

,id_deputado,deputado,presencas,sessoes_total,votacoes_participadas,taxa_votacao
0,62881,Danilo Forte,266,392,1827,0.303136
1,66179,NaN,221,392,1248,0.207068
2,66385,Julio Arcoverde,120,392,519,0.086112
3,66828,Fausto Pinato,329,392,2028,0.336486
4,67138,NaN,197,392,1093,0.181351
